# P-1 Random Forest Regressor with 5-Lag Telemetry and Context History

This notebook fits the final P-1 Random Forest forecasting model using the previous **5 observations** of both telemetry and context.

The earlier one-lag Random Forest used

$$
X_t =
[y_{t-1}, \mathbf{z}_{t-1}],
$$

where $\mathbf{z}_{t-1}$ contains the 24 context/command features.

The final RF(5) model uses

$$
X_t =
[
y_{t-1}, \mathbf{z}_{t-1},
y_{t-2}, \mathbf{z}_{t-2},
\ldots,
y_{t-5}, \mathbf{z}_{t-5}
].
$$

Because each lag contributes 1 telemetry variable and 24 context variables, the model has

$$
5(1+24)=125
$$

predictors.

The lag length was evaluated separately using training-only time-series cross-validation. Although lag 10 produced the numerically lowest mean validation MAE, lag 5 was within one standard error of that minimum and was retained as the simpler specification.

After the Random Forest produces telemetry forecasts, the anomaly-detection stage is kept fixed using the shared `detection.py` Telemanom post-prediction procedure.

**Important:** the Random Forest lag length and the Telemanom `history_length` parameter are different concepts.

- Random Forest history supplied to the forecaster: **5 lags**
- Telemanom detection `history_length`: **250**, retained to preserve the common P-1 prediction alignment and detector behavior


## 1. Import packages and locate the repository

This notebook is intended to run from the repository root or from its `Models` folder.


In [ ]:
# Standard libraries
import ast
import sys
from pathlib import Path

# Data and plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Random Forest regression and validation
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

# ------------------------------------------------------------
# Locate the repository root.
# ------------------------------------------------------------
repo_dir = Path.cwd().resolve()

if not (repo_dir / "detection.py").is_file():
    repo_dir = repo_dir.parent

if not (repo_dir / "detection.py").is_file():
    raise FileNotFoundError(
        "Could not find detection.py. Run this notebook from the "
        "fall-2026-spacecraft-sensors repository or its Models folder."
    )

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

import detection

print("Repository root:", repo_dir)
print("Detection module:", repo_dir / "detection.py")


## 2. Load P-1 data and anomaly labels

For P-1:

- column 0 is the telemetry signal,
- columns 1–24 are the context/command variables.

The official training set is used for model fitting and time-series cross-validation. The labeled anomaly intervals are not used in model fitting or hyperparameter selection.


In [ ]:
data_dir = repo_dir / "data" / "data"

p1_train = np.load(data_dir / "train" / "P-1.npy")
p1_test = np.load(data_dir / "test" / "P-1.npy")
labels = pd.read_csv(data_dir / "labeled_anomalies.csv")

print("Training shape:", p1_train.shape)
print("Test shape:", p1_test.shape)
print("Telemetry columns:", 1)
print("Context columns:", p1_train.shape[1] - 1)


## 3. Define the 5-lag predictor construction

For each target telemetry value $y_t$, the Random Forest receives the previous 5 complete P-1 rows:

$$
[
y_{t-1},z_{1,t-1},\ldots,z_{24,t-1},
y_{t-2},z_{1,t-2},\ldots,z_{24,t-2},
\ldots,
y_{t-5},z_{1,t-5},\ldots,z_{24,t-5}
].
$$

This creates

$$
5\times 25 = 125
$$

predictors.

The function below constructs these lagged predictors using only observations that occur before the target time.


In [ ]:
N_LAGS = 5
N_CONTEXT = p1_train.shape[1] - 1

def make_lagged_matrix(data, target_indices, n_lags):
    """Construct lagged predictors for selected target indices.

    Each target at time t receives the full row of predictors from
    t-1, t-2, ..., t-n_lags.

    Column 0 is telemetry and columns 1: are context variables.
    """
    target_indices = np.asarray(target_indices)

    if np.min(target_indices) < n_lags:
        raise ValueError(
            "Every target index must be at least n_lags."
        )

    lag_blocks = []
    predictor_names = []

    for lag in range(1, n_lags + 1):
        lag_blocks.append(data[target_indices - lag, :])

        predictor_names.append(f"telemetry_lag{lag}")
        predictor_names.extend([
            f"context_{j}_lag{lag}"
            for j in range(1, data.shape[1])
        ])

    X = np.hstack(lag_blocks)

    return X, predictor_names


# Training targets begin at index N_LAGS so all prior lagged rows exist.
train_target_indices = np.arange(
    N_LAGS,
    len(p1_train)
)

X_train, predictor_names = make_lagged_matrix(
    p1_train,
    train_target_indices,
    N_LAGS
)

y_train = p1_train[
    train_target_indices,
    0
]

print("Number of lags:", N_LAGS)
print("Training predictor matrix shape:", X_train.shape)
print("Training response shape:", y_train.shape)
print("Number of predictors:", X_train.shape[1])

assert X_train.shape[1] == N_LAGS * p1_train.shape[1]
assert X_train.shape[0] == len(y_train)


## 4. Why use 5 lags?

The earlier one-lag Random Forest received only the immediately previous telemetry/context observation.

Adding several lags allows the model to represent short-term temporal patterns such as:

- telemetry rising or falling over several observations,
- short oscillatory patterns,
- recent changes in context variables,
- nonlinear interactions between recent telemetry history and context history.

A Random Forest does not have recurrent memory like an LSTM, so temporal history must be supplied explicitly as lagged predictor columns.

Lag length was evaluated separately using training-only time-series cross-validation. Lag 10 had the lowest mean validation MAE, but lag 5 was within one standard error of the minimum and therefore provided a simpler model with statistically comparable forecasting performance.

The final model therefore uses **5 lags**, corresponding to 125 predictors.


## 5. Random Forest candidate settings

To make this experiment comparable with the earlier one-lag Random Forest, the same small hyperparameter grid is used.

The number of trees is fixed at 200 during cross-validation.

The candidate settings vary:

- `max_depth`,
- `min_samples_leaf`,
- `max_features`.

The anomaly labels are not used for hyperparameter selection.


In [ ]:
rf_candidates = [
    {
        "max_depth": 5,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    {
        "max_depth": 10,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    {
        "max_depth": 10,
        "min_samples_leaf": 5,
        "max_features": "sqrt"
    },
    {
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": "sqrt"
    },
    {
        "max_depth": 10,
        "min_samples_leaf": 5,
        "max_features": 1.0
    },
    {
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": 1.0
    }
]

print("Candidate configurations:", len(rf_candidates))


## 6. Time-series cross-validation

The candidate Random Forest configurations are compared using 5-fold `TimeSeriesSplit`.

This preserves temporal ordering: each validation block occurs after the training observations used to fit that fold.

The selection metric is mean absolute error:

$$
MAE =
\frac{1}{n}
\sum_{i=1}^{n}
|y_i-\hat y_i|.
$$

The configuration with the lowest mean validation MAE is selected.


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

cv_rows = []

for candidate_id, params in enumerate(
    rf_candidates,
    start=1
):
    for fold, (fit_idx, val_idx) in enumerate(
        tscv.split(X_train),
        start=1
    ):
        model_cv = RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1,
            **params
        )

        model_cv.fit(
            X_train[fit_idx],
            y_train[fit_idx]
        )

        val_predictions = model_cv.predict(
            X_train[val_idx]
        )

        fold_mae = mean_absolute_error(
            y_train[val_idx],
            val_predictions
        )

        cv_rows.append({
            "candidate": candidate_id,
            "fold": fold,
            "max_depth": params["max_depth"],
            "min_samples_leaf": params["min_samples_leaf"],
            "max_features": params["max_features"],
            "validation_MAE": fold_mae
        })

cv_results = pd.DataFrame(cv_rows)

display(cv_results.head(10))


## 7. Select the best Random Forest configuration

The candidate with the lowest mean validation MAE across the five time-series folds is selected.


In [ ]:
cv_summary = (
    cv_results
    .groupby(
        [
            "candidate",
            "max_depth",
            "min_samples_leaf",
            "max_features"
        ],
        dropna=False,
        as_index=False
    )
    .agg(
        mean_validation_MAE=("validation_MAE", "mean"),
        sd_validation_MAE=("validation_MAE", "std")
    )
    .sort_values("mean_validation_MAE")
    .reset_index(drop=True)
)

display(cv_summary)

best_candidate_id = int(
    cv_summary.loc[0, "candidate"]
)

best_params = rf_candidates[
    best_candidate_id - 1
]

print("Selected candidate:", best_candidate_id)
print("Selected parameters:", best_params)
print(
    "Mean validation MAE:",
    cv_summary.loc[0, "mean_validation_MAE"]
)


## 8. Fit the final 5-lag Random Forest

The selected Random Forest structural settings are fitted using all available 5-lag training observations.

The final model uses 300 trees.


In [ ]:
rf5_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    **best_params
)

rf5_model.fit(
    X_train,
    y_train
)

print("Final 5-lag Random Forest fitted.")
print("Number of trees:", len(rf5_model.estimators_))
print("Selected parameters:", best_params)


## 9. Inspect feature importance

With 500 predictors, looking only at individual feature importances can be difficult.

Two summaries are useful:

1. the most important individual lagged predictors,
2. total importance aggregated by lag.

If the model meaningfully uses longer history, some importance should appear beyond lag 1.


In [ ]:
# Individual feature importance.
importance_table = pd.DataFrame({
    "predictor": predictor_names,
    "importance": rf5_model.feature_importances_
})

# Add lag and feature-group metadata.
lag_values = []
feature_types = []

for lag in range(1, N_LAGS + 1):
    lag_values.extend(
        [lag] * p1_train.shape[1]
    )
    feature_types.extend(
        ["telemetry"]
        + [f"context_{j}" for j in range(1, N_CONTEXT + 1)]
    )

importance_table["lag"] = lag_values
importance_table["feature"] = feature_types

importance_table = (
    importance_table
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 individual lagged predictors:")
display(importance_table.head(20))

# Aggregate importance by lag.
importance_by_lag = (
    importance_table
    .groupby("lag", as_index=False)["importance"]
    .sum()
    .sort_values("lag")
)

display(importance_by_lag)

fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(
    importance_by_lag["lag"],
    importance_by_lag["importance"]
)

ax.set(
    title="P-1 Random Forest: total feature importance by lag",
    xlabel="Lag",
    ylabel="Total Random Forest importance"
)

ax.set_xticks(range(1, N_LAGS + 1))
plt.show()


## 10. Generate aligned P-1 test forecasts

To compare directly with the one-lag Random Forest, ARX, and the authors' saved LSTM predictions, the forecast range is kept unchanged:

- first forecasted test index: 250,
- last forecasted test index: 8494,
- number of forecasts: 8,245.

For every forecast at time $t$, the Random Forest now receives the previous 20 complete telemetry/context rows.

The fact that the first forecast is at test index 250 means all 20 required lags are available entirely within the test sequence.

Again, **20 is the Random Forest lag length**, whereas **250 is the P-1 forecast/detection alignment used by Telemanom**.


In [ ]:
# Preserve the P-1 forecast alignment used in the other models.
HISTORY_LENGTH = 250
N_PREDICTIONS = 10

n_forecasts = (
    len(p1_test)
    - HISTORY_LENGTH
    - N_PREDICTIONS
)

prediction_indices = np.arange(
    HISTORY_LENGTH,
    HISTORY_LENGTH + n_forecasts
)

# Construct 5-lag test predictors for exactly the same target indices.
X_test, test_predictor_names = make_lagged_matrix(
    p1_test,
    prediction_indices,
    N_LAGS
)

assert predictor_names == test_predictor_names

rf5_predictions = rf5_model.predict(
    X_test
)

actual_aligned = p1_test[
    prediction_indices,
    0
]

rf5_test_mae = mean_absolute_error(
    actual_aligned,
    rf5_predictions
)

print("Number of forecasts:", len(rf5_predictions))
print("First forecasted test index:", prediction_indices[0])
print("Last forecasted test index:", prediction_indices[-1])
print("Aligned test MAE:", rf5_test_mae)


## 11. Plot actual and predicted telemetry


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    prediction_indices,
    actual_aligned,
    linewidth=0.8,
    label="Actual telemetry"
)

ax.plot(
    prediction_indices,
    rf5_predictions,
    linewidth=0.8,
    alpha=0.8,
    label="5-lag Random Forest forecast"
)

ax.set(
    title="P-1: 5-lag Random Forest actual versus predicted telemetry",
    xlabel="Original test time-step index",
    ylabel="Scaled telemetry value"
)

ax.legend()
plt.show()


## 12. Apply the same Telemanom post-prediction detection procedure

Nothing in `detection.py` is changed for this experiment.

The same settings used for the one-lag Random Forest and ARX are retained:

- `history_length = 250`,
- `n_predictions = 10`,
- `batch_size = 70`,
- `window_size = 30`,
- `smoothing_perc = 0.05`,
- `error_buffer = 100`,
- `p = 0.13`,
- `initial_adjustment = True`,
- `initial_ignore = 500`.

Therefore, the intended experimental change is the amount of temporal history supplied to the Random Forest forecaster.


In [ ]:
rf5_detection = detection.detect_anomalies(
    test_data=p1_test,
    predictions=rf5_predictions,
    history_length=HISTORY_LENGTH,
    n_predictions=N_PREDICTIONS,
    target_column=0,
    batch_size=70,
    window_size=30,
    smoothing_perc=0.05,
    error_buffer=100,
    p=0.13,
    initial_adjustment=True,
    initial_ignore=500
)

print("Detected 5-lag Random Forest alert intervals:")
print(rf5_detection["detected_intervals"])

print()
print(
    "Number of detected intervals:",
    len(rf5_detection["detected_intervals"])
)


## 13. Examine raw and smoothed forecast errors

The raw forecast error is

$$
e_t = |y_t-\hat y_t|.
$$

The Telemanom post-processing procedure operates on smoothed versions of these errors.


In [ ]:
error_indices = rf5_detection["prediction_indices"]

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    error_indices,
    rf5_detection["raw_errors"],
    linewidth=0.7,
    alpha=0.5,
    label="Raw absolute error"
)

ax.plot(
    error_indices,
    rf5_detection["smoothed_errors"],
    linewidth=1.1,
    label="Smoothed error"
)

ax.set(
    title="P-1: 5-lag Random Forest forecast errors",
    xlabel="Original test time-step index",
    ylabel="Absolute prediction error"
)

ax.legend()
plt.show()


## 14. Load the labeled P-1 anomaly events

The anomaly labels are introduced only after the forecasting model and detector have produced their results.


In [ ]:
p1_label_rows = labels.loc[
    labels["chan_id"] == "P-1"
]

if p1_label_rows.empty:
    raise ValueError("No anomaly labels found for P-1.")

labeled_intervals = sorted({
    (int(start), int(end))
    for sequences in p1_label_rows["anomaly_sequences"]
    for start, end in ast.literal_eval(sequences)
})

print("Labeled P-1 events:")
print(labeled_intervals)

print()
print(
    "Anomaly class information:",
    p1_label_rows["class"].tolist()
)


## 15. Event-level anomaly detection evaluation

A labeled event counts as detected if at least one predicted alert interval overlaps it.

The main metrics are:

$$
\text{Precision}
=
\frac{TP}{TP+FP},
$$

$$
\text{Recall}
=
\frac{TP}{TP+FN},
$$

and

$$
F_1
=
2
\frac{\text{Precision}\times\text{Recall}}
{\text{Precision}+\text{Recall}}.
$$

We also calculate $F_{0.5}$, which weights precision more heavily.


In [ ]:
evaluation = detection.evaluate_event_detection(
    detected_intervals=rf5_detection["detected_intervals"],
    labeled_intervals=labeled_intervals
)

precision = evaluation["precision"]
recall = evaluation["recall"]

beta = 0.5

if np.isnan(precision) or np.isnan(recall):
    f_beta = np.nan
elif beta**2 * precision + recall == 0:
    f_beta = np.nan
else:
    f_beta = (
        (1 + beta**2)
        * precision
        * recall
        / (beta**2 * precision + recall)
    )

print("5-lag Random Forest + Telemanom detection")
print("-------------------------------------------")
print(f"TP: {evaluation['TP']}")
print(f"FP: {evaluation['FP']}")
print(f"FN: {evaluation['FN']}")

if np.isnan(precision):
    print("Precision: undefined (no detected intervals)")
else:
    print(f"Precision: {precision:.1%}")

print(f"Recall: {recall:.1%}")
print(f"F1: {evaluation['f1']:.3f}")

if np.isnan(f_beta):
    print("F0.5: undefined")
else:
    print(f"F0.5: {f_beta:.3f}")


## 16. Check which labeled events were detected


In [ ]:
event_table = pd.DataFrame({
    "labeled_interval": labeled_intervals,
    "detected": evaluation["detected_event_mask"]
})

display(event_table)


## 17. Check which detector branches retained readings

The Telemanom-style detector evaluates both:

- `high errors`,
- `low errors` (the inverted-error branch).

Only windows that retained at least one reading are shown below.


In [ ]:
active_windows = rf5_detection["window_results"].loc[
    rf5_detection["window_results"]["retained readings"] > 0
].copy()

display(active_windows)


## 18. Plot labeled events and detected alerts


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    np.arange(len(p1_test)),
    p1_test[:, 0],
    linewidth=0.8,
    label="Actual telemetry"
)

for i, (start, end) in enumerate(labeled_intervals):
    ax.axvspan(
        start,
        end,
        color= "green",
        alpha=0.18,
        label="Labeled event" if i == 0 else None
    )

for i, (start, end) in enumerate(
    rf5_detection["detected_intervals"]
):
    ax.axvspan(
        start,
        end,
        color = "red",
        alpha=0.10,
        hatch="//",
        label="RF-5 detected alert" if i == 0 else None
    )

ax.set(
    title="P-1: 5-lag Random Forest with Telemanom detection",
    xlabel="Original test time-step index",
    ylabel="Scaled telemetry value"
)

ax.legend()
plt.show()


# Interpretation

The final Random Forest model uses **5 lags** of telemetry and all 24 context variables:

$$
\text{RF(5): }
[y_{t-1},\mathbf z_{t-1},\ldots,y_{t-5},\mathbf z_{t-5}]
\longrightarrow
\hat y_t.
$$

This gives

$$
5\times25=125
$$

predictors.

The lag length was selected separately using training-only time-series cross-validation. Lag 10 produced the smallest mean validation MAE, but lag 5 was within one standard error of that minimum and was retained as the simpler specification.

The downstream anomaly detector is unchanged. Therefore, the final comparison is:

| Model | Temporal information supplied to forecaster | P-1 event-detection result |
|---|---|---|
| One-lag Random Forest | 1 telemetry lag + 1 context lag | 0/3 events in the earlier run |
| **RF(5)** | 5 telemetry lags + 5 context lags | **2/3 events, 0 false alerts** |
| Original Telemanom LSTM | long sequence history | 3/3 events under the shared corrected detector |

For RF(5), the first detected alert overlaps the first labeled event but extends beyond its labeled boundaries. Under the project's event-level overlap criterion, it is still counted as a true positive. The second labeled event is also detected, while the third is missed.

An important result from the lag experiments is that lower forecasting MAE did not automatically produce better anomaly detection. In particular, the RF model using 10 lags achieved slightly lower training-CV forecasting error than RF(5), but it did not detect the P-1 anomaly events under the fixed Telemanom detector.

This illustrates the distinction between the two objectives:

$$
\text{forecasting accuracy}
\neq
\text{anomaly-detection effectiveness}.
$$

Telemanom operates on the structure of the forecast residuals, so a forecasting model that predicts anomalous behavior too closely may generate residuals that are not unusual enough to trigger the detector.

## Important limitation

This is still not a fully controlled architecture comparison with the original LSTM.

RF(5) receives five explicitly constructed lagged rows, whereas the original LSTM uses a much longer sequential input and learns an internal nonlinear temporal representation.

Therefore, performance differences may reflect both:

1. the forecasting architecture, and
2. the amount and representation of temporal history.
